# Food Image Classification — ResNet18 vs ViT-Tiny

Multi-class image classification across **14 food categories** using two architectures compared side-by-side:
- **ResNet18** — frozen backbone, head-only training
- **ViT-Tiny** (patch16×16) — two-phase: head-only → full fine-tuning

## Classes
`Baked Potato` · `Crispy Chicken` · `Donut` · `Fries` · `Hot Dog` · `Sandwich` · `Taco` · `Taquito` · `apple_pie` · `cheesecake` · `chicken_curry` · `ice_cream` · `omelette` · `sushi`

## Dataset structure expected
```
DATA_DIR/
├── train/
├── val/
└── test/
    └── <class_name>/  (one folder per class)
```

## Key techniques
- Transfer learning with ImageNet-pretrained backbones (via `timm`)
- Two-phase training for ViT: head-only → full fine-tuning
- Early stopping: patience + train/val accuracy gap threshold
- Best-model checkpoint
- `CosineAnnealingLR` scheduler

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Set DATA_DIR to the folder containing train/, val/, test/ subdirectories
# Google Colab: '/content/drive/MyDrive/dataset_food_classification/dataset'
DATA_DIR = '.'  # <-- change this to your dataset path

train_dir = os.path.join(DATA_DIR, 'train')
val_dir   = os.path.join(DATA_DIR, 'val')
test_dir  = os.path.join(DATA_DIR, 'test')

os.makedirs('assets', exist_ok=True)

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    # Val e Test: solo resize a 224x224, nessun CenterCrop (non si scartano info ai bordi)
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

image_datasets = {
    'train': datasets.ImageFolder(train_dir, data_transforms['train']),
    'val':   datasets.ImageFolder(val_dir,   data_transforms['val']),
    'test':  datasets.ImageFolder(test_dir,  data_transforms['test'])
}

dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=32, shuffle=(x == 'train'),
                  num_workers=2, pin_memory=True)
    for x in ['train', 'val', 'test']
}

class_names = image_datasets['train'].classes
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Classi ({len(class_names)}): {class_names}')
print(f'Device: {device}')

## Visualizzazione Dataset e Data Augmentation

Prima del training visualizziamo:
1. **Campioni grezzi**: una immagine per classe selezionata casualmente.
2. **Effetto delle augmentation**: la stessa immagine trasformata più volte.

Augmentation applicate al **training set**:
- `RandomResizedCrop(224)` — ritaglia casualmente e ridimensiona a 224×224
- `RandomHorizontalFlip()` — flip orizzontale casuale
- `RandomRotation(±15°)` — piccola rotazione casuale
- Normalizzazione con media e std di ImageNet

**Val e Test**: solo `Resize((224, 224))`, senza `CenterCrop` per non scartare informazioni ai bordi.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import random

def denormalize(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    t = tensor.clone()
    for c, m, s in zip(range(3), mean, std):
        t[c] = t[c] * s + m
    return t.clamp(0, 1)

# 1. Immagini grezze dal dataset (7 classi casuali)
random.seed(42)
n_show = 7
selected_classes = random.sample(class_names, n_show)

fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3))
for ax, cls in zip(axes, selected_classes):
    cls_dir = os.path.join(train_dir, cls)
    img_file = random.choice(os.listdir(cls_dir))
    img = Image.open(os.path.join(cls_dir, img_file)).convert('RGB')
    ax.imshow(img)
    ax.set_title(cls, fontsize=8)
    ax.axis('off')
plt.suptitle('Campioni grezzi dal dataset (train)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('assets/sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

# 2. Stessa immagine con diverse applicazioni del transform di training
sample_cls = selected_classes[0]
sample_dir = os.path.join(train_dir, sample_cls)
random.seed(None)
sample_file = random.choice(os.listdir(sample_dir))
raw_img = Image.open(os.path.join(sample_dir, sample_file)).convert('RGB')

n_aug = 6
fig, axes = plt.subplots(1, n_aug + 1, figsize=(3 * (n_aug + 1), 3))
axes[0].imshow(raw_img)
axes[0].set_title('Originale', fontsize=9)
axes[0].axis('off')
for i in range(1, n_aug + 1):
    aug_tensor = data_transforms['train'](raw_img)
    aug_img = denormalize(aug_tensor).permute(1, 2, 0).numpy()
    axes[i].imshow(aug_img)
    axes[i].set_title(f'Aug #{i}', fontsize=9)
    axes[i].axis('off')
plt.suptitle(f'Data Augmentation — classe: "{sample_cls}"', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('assets/augmentation.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
for x in ['train', 'val', 'test']:
    print(f"{x.capitalize()} loader: {len(dataloaders[x])} batches (batch_size=32)")

inputs, labels = next(iter(dataloaders['test']))
print(f"\nBatch su CPU: {inputs.device}  →  GPU: {inputs.to(device).device}")

# Modello 1 — ResNet18 (Frozen Backbone)

Carichiamo ResNet18 pre-addestrata su ImageNet tramite `timm` e congelia l'intera backbone: viene allenato solo il layer classificatore finale. Questo approccio sfrutta le feature già apprese (bordi, texture, forme) riducendo i parametri trainabili e il rischio di overfitting.

| Parametro | Valore |
|-----------|--------|
| Ottimizzatore | Adam, lr=1e-3 |
| Scheduler | CosineAnnealingLR |
| Epoche | 20 (max) |
| Early stopping | patience=5 · gap>0.12 |

In [ ]:
import timm

EPOCHS = 20

model_ft = timm.create_model('resnet18', pretrained=True, num_classes=len(class_names))
model_ft = model_ft.to(device)

# Congela tutti i layer tranne il classificatore
for param in model_ft.parameters():
    param.requires_grad = False
for param in model_ft.get_classifier().parameters():
    param.requires_grad = True

criterion     = nn.CrossEntropyLoss()
optimizer_ft  = optim.Adam(model_ft.get_classifier().parameters(), lr=1e-3)
scheduler     = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=EPOCHS)

trainable = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_ft.parameters())
print(f'Parametri trainabili: {trainable:,} / {total:,}')

In [ ]:
import copy
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

def train_model(model, criterion, optimizer, scheduler, num_epochs=EPOCHS,
                patience=5, max_gap=0.15, checkpoint_path='best_model_checkpoint.pth'):
    """Training loop with early stopping (patience + accuracy gap) and best-model checkpoint."""
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc':  [], 'val_acc':  [],
        'train_f1':   [], 'val_f1':   []
    }
    best_model_wts  = copy.deepcopy(model.state_dict())
    best_acc        = 0.0
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)
        current_epoch_metrics = {}

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_loss, running_corrects = 0.0, 0
            all_labels, all_preds = [], []

            pbar = tqdm(dataloaders[phase], unit='batch', desc=f'{phase.capitalize()} Phase')
            for inputs, labels in pbar:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                running_loss     += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                temp_acc = running_corrects.double() / len(all_labels)
                temp_f1  = f1_score(all_labels, all_preds, average='macro')
                pbar.set_postfix(loss=loss.item(), acc=f'{temp_acc.item():.4f}', f1=f'{temp_f1:.4f}')

            if phase == 'train' and scheduler is not None:
                scheduler.step()

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc  = running_corrects.double() / len(image_datasets[phase])
            epoch_f1   = f1_score(all_labels, all_preds, average='macro')

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())
            history[f'{phase}_f1'].append(epoch_f1)
            current_epoch_metrics[phase] = epoch_acc.item()

            print(f'{phase} Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}  F1: {epoch_f1:.4f}')

            if phase == 'val':
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0
                    torch.save(model.state_dict(), checkpoint_path)
                else:
                    epochs_no_improve += 1

        gap = current_epoch_metrics['train'] - current_epoch_metrics['val']
        if gap > max_gap:
            print(f'Early Stopping: Accuracy Gap ({gap:.4f}) > soglia ({max_gap})')
            break
        if epochs_no_improve >= patience:
            print(f'Early Stopping: nessun miglioramento per {patience} epoche.')
            break

    print(f'Best val Acc: {best_acc:.4f}')
    model.load_state_dict(best_model_wts)
    return model, history


model_ft, history = train_model(
    model_ft, criterion, optimizer_ft, scheduler,
    num_epochs=EPOCHS, patience=5, max_gap=0.12,
    checkpoint_path='best_model_checkpoint.pth'
)

In [ ]:
def plot_history(history, title_suffix='', save_path=None):
    epochs = range(len(history['train_acc']))
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, (train_key, val_key), ylabel in zip(
        axes,
        [('train_loss', 'val_loss'), ('train_acc', 'val_acc'), ('train_f1', 'val_f1')],
        ['Loss', 'Accuracy', 'F1-Score']
    ):
        ax.plot(epochs, history[train_key], 'r', label='Train')
        ax.plot(epochs, history[val_key],   'b', label='Validation')
        ax.set_title(f'{ylabel} {title_suffix}')
        ax.set_xlabel('Epochs')
        ax.set_ylabel(ylabel)
        ax.grid(True)
        ax.legend()

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()

plot_history(history, title_suffix='— ResNet18', save_path='assets/training_history_resnet18.png')

# Modello 2 — Vision Transformer (ViT-Tiny)

Architettura basata su **self-attention** (Transformer) invece di convoluzioni: divide l'immagine in patch 16×16 e le processa con un encoder Transformer.

## Strategia di training in due fasi (gradual unfreezing)

| Fase | Cosa si allena | LR | Epoche |
|------|----------------|----|--------|
| 1 — Head-only | Solo il classificatore finale | 1e-3 | 10 |
| 2 — Full fine-tuning | Tutta la rete (backbone + head) | 1e-4 | 20 |

**Motivazione**: nella fase 1 la backbone è congelata e si impara rapidamente a mappare le feature ViT sulle classi. Nella fase 2 si sblocca l'intera rete con LR molto basso per adattare le rappresentazioni interne senza distruggere il pretraining (catastrophic forgetting).

In [ ]:
EPOCHS_HEAD     = 10
EPOCHS_FINETUNE = 20

model_vit = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=len(class_names))
model_vit = model_vit.to(device)

# Congela tutta la backbone, sblocca solo la testa
for param in model_vit.parameters():
    param.requires_grad = False
for param in model_vit.get_classifier().parameters():
    param.requires_grad = True

criterion_vit = nn.CrossEntropyLoss()
optimizer_vit = optim.Adam(model_vit.get_classifier().parameters(), lr=1e-3)
scheduler_vit = optim.lr_scheduler.CosineAnnealingLR(optimizer_vit, T_max=EPOCHS_HEAD)

trainable = sum(p.numel() for p in model_vit.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_vit.parameters())
print(f'Parametri trainabili (Fase 1): {trainable:,} / {total:,}')

In [ ]:
# Fase 1: head-only
print('=' * 60)
print(f'FASE 1 — Head-only  (lr=1e-3, max {EPOCHS_HEAD} epoche)')
print('=' * 60)
model_vit, history_vit_p1 = train_model(
    model_vit, criterion_vit, optimizer_vit, scheduler_vit,
    num_epochs=EPOCHS_HEAD, patience=5, max_gap=0.12,
    checkpoint_path='best_vit_checkpoint.pth'
)

# Fase 2: full fine-tuning
print()
print('=' * 60)
print(f'FASE 2 — Full fine-tuning  (lr=1e-4, max {EPOCHS_FINETUNE} epoche)')
print('=' * 60)

for param in model_vit.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model_vit.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_vit.parameters())
print(f'Parametri trainabili (Fase 2): {trainable:,} / {total:,}')

optimizer_vit_ft = optim.Adam(model_vit.parameters(), lr=1e-4)
scheduler_vit_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_vit_ft, T_max=EPOCHS_FINETUNE)

model_vit, history_vit_p2 = train_model(
    model_vit, criterion_vit, optimizer_vit_ft, scheduler_vit_ft,
    num_epochs=EPOCHS_FINETUNE, patience=5, max_gap=0.12,
    checkpoint_path='best_vit_checkpoint.pth'
)

# Unione delle history delle due fasi
history_vit = {k: history_vit_p1[k] + history_vit_p2[k] for k in history_vit_p1}
print(f'Training completato. Epoche totali: {len(history_vit["train_acc"])}')

In [ ]:
p2_start = len(history_vit_p1['train_acc'])

epochs = range(len(history_vit['train_acc']))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (train_key, val_key), ylabel in zip(
    axes,
    [('train_loss', 'val_loss'), ('train_acc', 'val_acc'), ('train_f1', 'val_f1')],
    ['Loss', 'Accuracy', 'F1-Score']
):
    ax.plot(epochs, history_vit[train_key], 'r', label='Train')
    ax.plot(epochs, history_vit[val_key],   'b', label='Validation')
    ax.axvline(p2_start - 0.5, color='green', linestyle='--', linewidth=1.5, label='Inizio Fase 2')
    ax.set_title(f'{ylabel} — ViT-Tiny')
    ax.set_xlabel('Epochs')
    ax.set_ylabel(ylabel)
    ax.grid(True)
    ax.legend()

plt.tight_layout()
plt.savefig('assets/training_history_vit.png', dpi=120, bbox_inches='tight')
plt.show()

Con il cambio di architettura e la strategia di training in due fasi si riscontrano notevoli miglioramenti. Verso la fine del training è visibile un tipico segnale di lieve overfitting (val_loss circa costante, train_loss in discesa), accettabile considerando l'elevato numero di classi.

# Valutazione Finale

Metriche complete (Precision, Recall, F1, Accuracy) e Confusion Matrix su **train** e **test set** per entrambe le architetture.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns

def evaluate_model_detailed(model, dataloader, classes, split_name='Test', save_path=None):
    """Accuracy, classification report e confusion matrix su un dataloader."""
    model.eval()
    all_labels, all_preds = [], []

    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc=f'Evaluating {split_name}'):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    overall_acc = accuracy_score(all_labels, all_preds)
    print(f'\n[{split_name}] Accuracy: {overall_acc:.4f} ({overall_acc*100:.2f}%)')
    print(f'\nClassification Report ({split_name}):')
    print(classification_report(all_labels, all_preds, target_names=classes, digits=4))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes,
                cmap='Blues', linewidths=0.5)
    plt.title(f'Confusion Matrix — {split_name}', fontsize=14)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()

    return overall_acc, all_labels, all_preds

## ResNet18 — Valutazione

In [ ]:
model_ft_eval = timm.create_model('resnet18', pretrained=False, num_classes=len(class_names))
model_ft_eval.load_state_dict(torch.load('best_model_checkpoint.pth', map_location=device, weights_only=True))
model_ft_eval = model_ft_eval.to(device)

print('=' * 60)
print('RESNET18 — TRAINING SET')
print('=' * 60)
acc_resnet_train, _, _ = evaluate_model_detailed(
    model_ft_eval, dataloaders['train'], class_names,
    split_name='Train (ResNet18)', save_path='assets/confusion_matrix_resnet18_train.png')

print('=' * 60)
print('RESNET18 — TEST SET')
print('=' * 60)
acc_resnet_test, _, _ = evaluate_model_detailed(
    model_ft_eval, dataloaders['test'], class_names,
    split_name='Test (ResNet18)', save_path='assets/confusion_matrix_resnet18_test.png')

## ViT-Tiny — Valutazione

In [ ]:
print('=' * 60)
print('VIT-TINY — TRAINING SET')
print('=' * 60)
acc_vit_train, _, _ = evaluate_model_detailed(
    model_vit, dataloaders['train'], class_names,
    split_name='Train (ViT-Tiny)', save_path='assets/confusion_matrix_vit_train.png')

print('=' * 60)
print('VIT-TINY — TEST SET')
print('=' * 60)
acc_vit_test, _, _ = evaluate_model_detailed(
    model_vit, dataloaders['test'], class_names,
    split_name='Test (ViT-Tiny)', save_path='assets/confusion_matrix_vit_test.png')

In [ ]:
models_labels = ['ResNet18\n(Train)', 'ResNet18\n(Test)', 'ViT-Tiny\n(Train)', 'ViT-Tiny\n(Test)']
accuracies    = [acc_resnet_train, acc_resnet_test, acc_vit_train, acc_vit_test]
colors        = ['#4C72B0', '#DD8452', '#4C72B0', '#DD8452']

plt.figure(figsize=(9, 5))
bars = plt.bar(models_labels, accuracies, color=colors, edgecolor='black', width=0.5)
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f'{acc:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.ylim(0, 1.1)
plt.ylabel('Accuracy', fontsize=13)
plt.title('Confronto Accuracy: ResNet18 vs ViT-Tiny (Train vs Test)', fontsize=13)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('assets/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nRiepilogo finale:')
print(f'  ResNet18 -> Train Acc: {acc_resnet_train:.4f} | Test Acc: {acc_resnet_test:.4f}')
print(f'  ViT-Tiny -> Train Acc: {acc_vit_train:.4f} | Test Acc: {acc_vit_test:.4f}')
best = 'ResNet18' if acc_resnet_test >= acc_vit_test else 'ViT-Tiny'
print(f'\n=> Modello migliore sul Test Set: {best}')

---
# Riepilogo del Progetto

## Obiettivo
Classificazione di immagini di cibo su **14 classi** con train/val/test split.

## Pre-processing e Data Augmentation

**Training set**: `RandomResizedCrop(224)` · `RandomHorizontalFlip()` · `RandomRotation(±15°)` · Normalizzazione ImageNet.

**Val/Test**: solo `Resize((224, 224))` senza `CenterCrop` — scelta deliberata per non scartare informazioni ai bordi.

## Architetture Testate

| Modello | Parametri | Strategia |
|---------|-----------|----------|
| ResNet18 | ~11M | Backbone congelata, solo head |
| ViT-Tiny | ~5.5M | Fase 1: head · Fase 2: full fine-tuning |

## Strategia di Training
- **Early Stopping** con due criteri indipendenti: *patience = 5* e *accuracy gap > 0.12*
- **Checkpoint automatico** del modello con miglior validation accuracy
- **CosineAnnealingLR** come scheduler in tutte le fasi

## Conclusioni
Il ViT-Tiny con training in due fasi supera significativamente ResNet18 in tutte le metriche. Il transfer learning con backbone congelata nella prima fase previene il catastrophic forgetting e accelera la convergenza. L'early stopping basato sull'accuracy gap ha efficacemente limitato l'overfitting.